In [0]:
# 01_synthetic_data — seed 파일을 ADLS raw/ 에 복사
# DDL은 00_apply_migrations.py 에서 처리한다.

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("storage_account", "acdriftadls")
dbutils.widgets.text("raw_base", "abfss://lakehouse@acdriftadls.dfs.core.windows.net/raw/dev")
dbutils.widgets.text("seed_data_path", "data/dev/landing/raw")

CATALOG = dbutils.widgets.get("catalog")
STORAGE = dbutils.widgets.get("storage_account")
RAW_BASE_PARAM = dbutils.widgets.get("raw_base")
SEED_DATA_PATH = dbutils.widgets.get("seed_data_path").strip("/")

RAW_BASE = RAW_BASE_PARAM or f"abfss://lakehouse@{STORAGE}.dfs.core.windows.net/raw"
SEED_BASE = f"/Workspace/Shared/access-drift/files/{SEED_DATA_PATH}"


In [0]:
import os

if not os.path.isdir(SEED_BASE):
    raise RuntimeError(
        f"Seed 파일 디렉터리를 찾을 수 없습니다: {SEED_BASE}\n"
        "bundle deploy가 완료됐는지 확인하세요."
    )
print(f"seed path OK: {SEED_BASE}")

## Seed 파일 → ADLS raw/ 복사
# target별 seed_data_path를 ADLS raw_base 경로로 복사한다.
# 주의: 동일 raw_base의 같은 파일을 덮어쓰므로 target/catalog와 raw 경로를 확인한 뒤 실행한다.


In [0]:
seed_mappings = [
    ("azure/applications.jsonl",        f"{RAW_BASE}/azure/applications.jsonl"),
    ("azure/application_owners.jsonl",  f"{RAW_BASE}/azure/application_owners.jsonl"),
    ("azure/password_credentials.jsonl",f"{RAW_BASE}/azure/password_credentials.jsonl"),
    ("identity/lifecycle_seed.jsonl",   f"{RAW_BASE}/identity/lifecycle_seed.jsonl"),
    ("assets/assets_seed.jsonl",        f"{RAW_BASE}/assets/assets_seed.jsonl"),
    ("access/access_edges_seed.jsonl",  f"{RAW_BASE}/access/access_edges_seed.jsonl"),
]

for rel_path, dst in seed_mappings:
    local_src = f"{SEED_BASE}/{rel_path}"
    if not os.path.isfile(local_src):
        raise RuntimeError(f"Seed 파일을 찾을 수 없습니다: {local_src}")

    dbutils.fs.mkdirs(dst.rsplit("/", 1)[0])
    dbutils.fs.cp(f"file:{local_src}", dst, recurse=False)
    print(f"copied: {rel_path} → {dst}")


## Smoke test

In [0]:
uploaded_files = []
for _, dst in seed_mappings:
    parent = dst.rsplit("/", 1)[0]
    filename = dst.rsplit("/", 1)[1]
    files = [f.name.rstrip("/") for f in dbutils.fs.ls(parent)]
    assert filename in files, f"raw seed file not found: {dst}"
    uploaded_files.append(dst)

summary = ["seed upload smoke test passed", f"uploaded_files={len(uploaded_files)}"]
summary.extend(uploaded_files)
print("\n".join(summary))
dbutils.notebook.exit("\n".join(summary))


seed upload smoke test passed
seed upload complete
